# Geographic data in Python - Spatial Data

Adapted from geocompy Chapter 1. All data paths adjusted for running from the `sam/` directory.

## Introduction

This chapter outlines two fundamental geographic data models (vector and raster) and introduces Python packages for working with them.

- The **vector data model** represents geographic entities with points, lines, and polygons.
- The **raster data model** divides the surface up into cells of constant size.

We will focus on **shapely** and **geopandas** for working with geographic vector data, and **rasterio** for working with rasters.

In [ ]:
# Setup: imports and display options
import matplotlib.pyplot as plt
import pandas as pd
import shapely
import geopandas as gpd
import numpy as np
import rasterio
import rasterio.plot
import pyproj

pd.set_option('display.max_rows', 6)
pd.set_option('display.max_columns', 5)
pd.options.display.max_colwidth = 35
plt.rcParams['figure.figsize'] = (5, 5)

# Data directory (relative to sam/)
DATA_DIR = '../data'
OUTPUT_DIR = '../output'

## Vector data

### Vector layers

The most commonly used geographic vector data structure is the vector layer. We use **geopandas** to work with them.

In [ ]:
gdf = gpd.read_file(f'{DATA_DIR}/world.gpkg')

In [ ]:
type(gdf)

In [ ]:
gdf.shape

The `GeoDataFrame` class is an extension of the `DataFrame` class from **pandas**. Standard data frame subsetting methods can be used.

In [ ]:
gdf = gdf[['name_long', 'geometry']]
gdf

The following expression creates a subdataset based on a condition:

In [ ]:
gdf[gdf['name_long'] == 'Egypt']

To get a sense of the spatial component, the vector layer can be plotted using `.plot`:

In [ ]:
gdf.plot();

Interactive maps can be created with `.explore`:

In [ ]:
gdf.explore()

A subset can also be plotted interactively:

In [ ]:
gdf[gdf['name_long'] == 'Egypt'].explore()

### Geometry columns

The geometry column of class `GeoSeries` is an essential column in a `GeoDataFrame`. It contains the geometric part of the vector layer.

In [ ]:
gdf.geometry

The geometry column also contains the spatial reference information:

In [ ]:
gdf.geometry.crs

Geometry operations like `.envelope` return a `GeoSeries` containing bounding box polygons:

In [ ]:
gdf.envelope

In [ ]:
gdf.geometry.envelope

To keep attributes alongside modified geometry, create a copy and overwrite the geometry:

In [ ]:
gdf2 = gdf.copy()
gdf2.geometry = gdf.envelope
gdf2

The geometry type property returns the type of each geometry:

In [ ]:
gdf.geometry.type

In [ ]:
gdf.geometry.type.value_counts()

A `GeoDataFrame` can also have multiple `GeoSeries` columns:

In [ ]:
gdf['bbox'] = gdf.envelope
gdf['polygon'] = gdf.geometry
gdf

Switch the active geometry column using `.set_geometry`:

In [ ]:
gdf = gdf.set_geometry('bbox')
gdf.explore()

In [ ]:
gdf = gdf.set_geometry('polygon')
gdf.explore()

### Geometries

Each element in the geometry column is a `shapely` geometry object.

In [ ]:
gdf.geometry.iloc[3]

In [ ]:
gdf[gdf['name_long'] == 'Egypt'].geometry.iloc[0]

### Creating shapely geometries

**Point:**

In [ ]:
point = shapely.Point([5, 2])
point

In [ ]:
# Alternatively from WKT
point = shapely.from_wkt('POINT (5 2)')
point

**LineString:**

In [ ]:
linestring = shapely.LineString([(1,5), (4,4), (4,1), (2,2), (3,2)])
linestring

**Polygon:**

In [ ]:
polygon = shapely.Polygon(
    [(1,5), (2,2), (4,1), (4,4), (1,5)],  ## Exterior
    [[(2,4), (3,4), (3,3), (2,3), (2,4)]]  ## Hole(s)
)
polygon

**MultiPoint:**

In [ ]:
multipoint = shapely.MultiPoint([(5,2), (1,3), (3,4), (3,2)])
multipoint

**MultiLineString:**

In [ ]:
multilinestring = shapely.MultiLineString([
    [(1,5), (4,4), (4,1), (2,2), (3,2)],  ## 1st sequence
    [(1,2), (2,4)]  ## 2nd sequence
])
multilinestring

**MultiPolygon:**

In [ ]:
multipolygon = shapely.MultiPolygon([
    [[(1,5), (2,2), (4,1), (4,4), (1,5)], []],  ## Polygon 1 
    [[(0,2), (1,2), (1,3), (0,3), (0,2)], []]   ## Polygon 2
])
multipolygon

In [ ]:
# Alternative: create Polygons first, then combine
multipolygon = shapely.MultiPolygon([
    shapely.Polygon([(1,5), (2,2), (4,1), (4,4), (1,5)]),
    shapely.Polygon([(0,2), (1,2), (1,3), (0,3), (0,2)])
])
multipolygon

**GeometryCollection:**

In [ ]:
geometrycollection = shapely.GeometryCollection([multipoint, multilinestring])
geometrycollection

### Geometry operations example

The difference between a buffered `MultiPolygon` and itself:

In [ ]:
multipolygon.buffer(0.2).difference(multipolygon)

Print the WKT representation:

In [ ]:
print(linestring)

Access raw coordinates of geometry exterior:

In [ ]:
list(polygon.exterior.coords)

### Vector layer from scratch

Constructing a `GeoDataFrame` from `shapely` geometries combined into a `GeoSeries`.

In [ ]:
lnd_point = shapely.Point(0.1, 51.5)
lnd_point

In [ ]:
lnd_geom = gpd.GeoSeries([lnd_point], crs=4326)
lnd_geom

In [ ]:
lnd_data = {
  'name': ['London'],
  'temperature': [25],
  'date': ['2023-06-21'],
  'geometry': lnd_geom
}
lnd_layer = gpd.GeoDataFrame(lnd_data)
lnd_layer

Creating a layer with multiple features (London and Paris):

In [ ]:
lnd_point = shapely.Point(0.1, 51.5)
paris_point = shapely.Point(2.3, 48.9)
towns_geom = gpd.GeoSeries([lnd_point, paris_point], crs=4326)
towns_data = {
  'name': ['London', 'Paris'],
  'temperature': [25, 27],
  'date': ['2013-06-21', '2013-06-21'],
  'geometry': towns_geom
}
towns_layer = gpd.GeoDataFrame(towns_data)
towns_layer

In [ ]:
towns_layer.explore(color='red', marker_kwds={'radius': 10})

Creating a spatial layer from a `DataFrame` with coordinate columns:

In [ ]:
towns_table = pd.DataFrame({
  'name': ['London', 'Paris'],
  'temperature': [25, 27],
  'date': ['2017-06-21', '2017-06-21'],
  'x': [0.1, 2.3],
  'y': [51.5, 48.9]
})
towns_geom = gpd.points_from_xy(towns_table['x'], towns_table['y'])
towns_layer = gpd.GeoDataFrame(towns_table, geometry=towns_geom, crs=4326)
towns_layer

### Derived numeric properties

Vector layers have two essential derived numeric properties: **length** (for lines) and **area** (for polygons).

In [ ]:
linestring.length

In [ ]:
multipolygon.area

In [ ]:
gpd.GeoSeries([point, linestring, polygon, multipolygon]).area

To get meaningful measurements, project to a suitable CRS first. For example, area of Slovenia in UTM zone 33N (m²):

In [ ]:
# Re-read world data with all columns for this example
gdf_full = gpd.read_file(f'{DATA_DIR}/world.gpkg')
gdf_full[gdf_full['name_long'] == 'Slovenia'].to_crs(32633).area

## Raster data

### Using rasterio

Importing a raster is a two-step process:
1. Open a raster file connection with `rasterio.open`
2. Read raster values with `.read`

In [ ]:
src = rasterio.open(f'{DATA_DIR}/srtm.tif')
src

In [ ]:
rasterio.plot.show(src);

The `DatasetReader` contains the raster metadata:

In [ ]:
src.meta

Read the actual raster values (first and only band):

In [ ]:
src.read(1)

### Raster from scratch

Creating rasters from scratch: `elev` (continuous) and `grain` (categorical).

In [ ]:
elev = np.arange(1, 37, dtype=np.uint8).reshape(6, 6)
elev

In [ ]:
v = [
  1, 0, 1, 2, 2, 2, 
  0, 2, 0, 0, 2, 1, 
  0, 2, 2, 0, 0, 2, 
  0, 0, 1, 1, 1, 1, 
  1, 1, 1, 2, 1, 1, 
  2, 1, 2, 2, 0, 2
]
grain = np.array(v, dtype=np.uint8).reshape(6, 6)
grain

Create the georeferencing transformation matrix:

In [ ]:
new_transform = rasterio.transform.from_origin(
    west=-1.5, 
    north=1.5, 
    xsize=0.5, 
    ysize=0.5
)
new_transform

In [ ]:
rasterio.plot.show(elev, transform=new_transform);

In [ ]:
rasterio.plot.show(grain, transform=new_transform);

Export the rasters to files:

In [ ]:
import os
os.makedirs('output', exist_ok=True)

new_dataset = rasterio.open(
    'output/elev.tif', 'w', 
    driver='GTiff',
    height=elev.shape[0],
    width=elev.shape[1],
    count=1,
    dtype=elev.dtype,
    crs=4326,
    transform=new_transform
)
new_dataset.write(elev, 1)
new_dataset.close()

new_dataset = rasterio.open(
    'output/grain.tif', 'w', 
    driver='GTiff',
    height=grain.shape[0],
    width=grain.shape[1],
    count=1,
    dtype=grain.dtype,
    crs=4326,
    transform=new_transform
)
new_dataset.write(grain, 1)
new_dataset.close()

print('Rasters saved to output/elev.tif and output/grain.tif')

## Coordinate Reference Systems (CRS)

CRSs define how spatial elements relate to the Earth's surface. They are either **geographic** (lon/lat) or **projected** (e.g., meters).

In [ ]:
epsg_codes = pyproj.get_codes('EPSG', 'CRS')
epsg_codes[:5]

In [ ]:
pyproj.CRS.from_epsg(4326)

Query the CRS of a vector layer:

In [ ]:
zion = gpd.read_file(f'{DATA_DIR}/zion.gpkg')
zion.crs

Compare geographic vs. projected CRS:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# WGS84 (geographic)
zion.to_crs(4326).plot(ax=axes[0], edgecolor='black', color='lightgrey')
axes[0].grid()
axes[0].set_title('Geographic (WGS84)')

# NAD83 / UTM zone 12N (projected)
zion.plot(ax=axes[1], edgecolor='black', color='lightgrey')
axes[1].grid()
axes[1].set_title('Projected (NAD83 / UTM zone 12N)')

plt.tight_layout();

## Units

Python spatial data structures do not natively support measurement units. The coordinates are plain numbers referring to the CRS.

In [ ]:
src.meta